# PubMedQA — Retrieval + Answer Quality Evaluation

End-to-end notebook for evaluating a RAG pipeline on [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA):

1. **Retrieval quality** — did we retrieve the correct paper context for each question?
2. **Answer quality** — does the LLM-generated answer match the reference `long_answer`? (ROUGE-1/2/L)
3. **LLM judge** — semantic quality scored by a local LLM

**Pipeline:**  
PubMedQA question → TTS audio → ASR → text embedding → corpus retrieval → **answer generation (Phase 4.5)** → ROUGE + LLM judge

All generation and judging is configured via `EvaluationConfig`. No inline LLM code needed.

**Requirements:** Ollama running locally (`http://localhost:11434`) with at least one model pulled.

## 1. Install

In [ ]:
%pip install -q -e '..'
%pip install -q datasets rouge-score

## 2. Load PubMedQA from HuggingFace

PubMedQA (`pqa_labeled`) has 1 000 expert-annotated samples. Each sample:
- `question` — a yes/no/maybe medical question about a paper  
- `context["contexts"]` — passages from that paper (the ground-truth context)  
- `long_answer` — author-written conclusion/answer (~2–5 sentences)  
- `final_decision` — `"yes"` / `"no"` / `"maybe"`

We convert to the evaluator's two-file format:
- **`questions.json`** — one entry per question, with `groundtruth_doc_ids = [str(pubid)]`
- **`corpus.json`** — one entry per `pubid`, text = joined context passages

In [ ]:
from pathlib import Path
import json

from datasets import load_dataset

N_SAMPLES = 50  # reduce for faster runs; 1000 = full labeled set

DATA_DIR       = Path("data/pubmedqa")
QUESTIONS_PATH = DATA_DIR / "questions.json"
CORPUS_PATH    = DATA_DIR / "corpus.json"
DATA_DIR.mkdir(parents=True, exist_ok=True)

hf_ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
hf_ds = hf_ds.select(range(min(N_SAMPLES, len(hf_ds))))
print(hf_ds)
print("\nSample row keys:", list(hf_ds[0].keys()))
print("context sub-keys:", list(hf_ds[0]["context"].keys()))

In [ ]:
questions = []
corpus    = []

for row in hf_ds:
    pubid   = str(row["pubid"])
    ctx     = row["context"]
    passages = ctx["contexts"]          # list[str]
    labels   = ctx["labels"]            # list[str]: BACKGROUND, RESULTS, …

    # corpus: one doc per paper — all labeled passages joined
    labeled_text = "\n\n".join(
        f"[{lbl}] {txt}" for lbl, txt in zip(labels, passages)
    )
    corpus.append({
        "doc_id":   pubid,
        "title":    row["question"],   # no separate title in dataset
        "text":     labeled_text,
        "metadata": {
            "long_answer":      row["long_answer"],
            "final_decision":   row["final_decision"],
            "meshes":           ctx["meshes"],
        },
    })

    # question: ground truth = the single matching pubid
    questions.append({
        "question_id":        f"q_{pubid}",
        "question_text":      row["question"],
        "groundtruth_doc_ids": [pubid],
        "relevance_grades":   {pubid: 1},
        "metadata": {
            "final_decision": row["final_decision"],
        },
    })

QUESTIONS_PATH.write_text(json.dumps(questions, indent=2))
CORPUS_PATH.write_text(json.dumps(corpus, indent=2))

print(f"Saved {len(questions)} questions → {QUESTIONS_PATH}")
print(f"Saved {len(corpus)} corpus docs  → {CORPUS_PATH}")
print()
print("Example question:", questions[0]["question_text"])
print("Groundtruth doc: ", questions[0]["groundtruth_doc_ids"])
print("Context preview: ", corpus[0]["text"][:200], "...")

## 3. Retrieval + Answer Generation Config

Run the complete evaluator pipeline:
- **TTS synthesis** — MMS auto-synthesizes audio for questions
- **ASR** — Whisper-tiny transcribes audio back to text
- **Text embedding** — Jina v4 encodes transcripts and corpus docs
- **Retrieval** — in-memory cosine search, top-k=5
- **Query optimization** (optional) — rewrite/HyDE/decompose/multi-query via local LLM
- **Phase 5: Answer Generation** — local Ollama generates answers from retrieved context
- **LLM Judge** — compares generated vs reference answer (`judge_mode="answer_quality"`)
- **Metrics** — MRR, Recall@k, NDCG@k, ROUGE-1/2/L

The `llm` top-level block sets the shared LLM backend (model, Ollama URL, API key env).  
`query_optimization`, `answer_generation`, and `judge` inherit from it — no duplication needed.

In [ ]:
import urllib.request as _ureq
import json as _json

from evaluator import EvaluationConfig, run_evaluation

OLLAMA_BASE     = "http://localhost:11434"
PREFERRED_MODEL = "gemma4"  # change to your preferred model


def _pick_ollama_model(preferred, available):
    if not available:
        return preferred
    if preferred in available:
        return preferred
    base = preferred.split(":")[0].lower()
    for name in available:
        if name.split(":")[0].lower() == base:
            return name
    return available[0]


try:
    with _ureq.urlopen(f"{OLLAMA_BASE}/api/tags", timeout=3) as _r:
        _available = [m["name"] for m in _json.loads(_r.read()).get("models", [])]
    _model = _pick_ollama_model(PREFERRED_MODEL, _available)
    print(f"[ollama] Using model: {_model!r}   (available: {_available})")
except Exception as _e:
    _model = PREFERRED_MODEL
    print(f"[ollama] Cannot connect to {OLLAMA_BASE}: {_e}")
    print("Sections 5-8 require Ollama running locally.")


cfg = EvaluationConfig.from_dict({
    "experiment_name": "pubmedqa_retrieval",
    "model": {
        "pipeline_mode":       "asr_text_retrieval",
        "asr_model_type":      "whisper",
        "asr_model_name":      "openai/whisper-tiny",
        "asr_device":          "cuda:0",
        "text_emb_model_type": "jina_v4",
        "text_emb_model_name": "jinaai/jina-embeddings-v4",
        "text_emb_device":     "cuda:0",
    },
    "data": {
        "questions_path": str(QUESTIONS_PATH),
        "corpus_path":    str(CORPUS_PATH),
        "trace_limit":    N_SAMPLES,
        "batch_size":     8,
    },
    "audio_synthesis": {
        "enabled":    True,
        "provider":   "mms",
        "output_dir": str(DATA_DIR / "audio"),
    },
    # --- shared LLM backend (judge, answer_generation, query_optimization all inherit) ---
    "llm": {
        "model":            _model,
        "use_local_server": True,
        "local_server_url": f"{OLLAMA_BASE}/v1/chat/completions",
        "api_key_env":      "OLLAMA_API_KEY",
    },
    "query_optimization": {
        "enabled":          True,          # set True to rewrite queries before retrieval
        "method":           "rewrite",      # rewrite | hyde | decompose | multi_query
        "combine_strategy": "rrf",          # for decompose / multi_query only
    },
    "answer_generation": {
        "enabled":                  True,
        "method":                   "simple",   # simple | chain_of_thought | multi_query
        "context_docs":             3,
        "context_max_chars":        600,
        "compute_rouge":            True,
        "reference_metadata_field": "long_answer",
    },
    "judge": {
        "enabled":    True,
        "judge_mode": "answer_quality",     # retrieval | answer_quality | both
        "max_cases":  0,                    # 0 = all traces
    },
    "vector_db": {"type": "inmemory", "k": 5},
    "cache": {
        "enabled":   True,
        "cache_dir": str(DATA_DIR / "cache"),
    },
    "service_runtime": {
        "startup_mode":   "lazy",
        "offload_policy": "on_finish",
    },
    "tracking": {"enabled": False},
}, validate=False)

In [ ]:
results = run_evaluation(cfg)
print(results)

## 4. Inspect Retrieval Traces

Per-query breakdown: question → expected doc → retrieved docs with scores.

In [ ]:
traces = results.get_metric("query_traces", [])
print(f"{len(traces)} query traces\n")

# Build quick lookup: doc_id → long_answer
_long_answers = {doc["doc_id"]: doc["metadata"]["long_answer"] for doc in corpus}

hits = 0
for t in traces[:10]:   # show first 10
    q_id     = t.get("query_id", "")
    relevant = set(t.get("relevant", {}).keys())
    retrieved_ids = [r["doc_key"] for r in t.get("retrieved", [])]
    hit = bool(relevant & set(retrieved_ids))
    if hit:
        hits += 1
    status = "✓ HIT" if hit else "✗ MISS"
    print(f"[{status}] {q_id}")
    print(f"  Expected: {relevant}")
    print(f"  Retrieved: {retrieved_ids}")
    print()

total_hits = sum(
    bool(set(t.get("relevant", {}).keys()) &
         {r["doc_key"] for r in t.get("retrieved", [])})
    for t in traces
)
print(f"Recall@5 (manual): {total_hits}/{len(traces)} = {total_hits/len(traces):.3f}")
print(f"MRR: {results.get_metric('MRR', 0):.4f}")

## 5. Answer Generation Results

Phase 4.5 ran inside the pipeline. Results (generated answers + ROUGE vs `long_answer`) are in `results.get_metric("answer_generation")`.

In [ ]:
ans_gen = results.get_metric("answer_generation", {})

print(f"Method:  {ans_gen.get('method')}")
print(f"Model:   {ans_gen.get('model')}")
print(f"Cases:   {ans_gen.get('cases')}")
print()

details = ans_gen.get("details", [])
if details:
    ex = details[0]
    print("Example:")
    print(f"  Q:         {ex['question']}")
    print(f"  Generated: {ex['generated_answer'][:200]}")
    print(f"  Reference: {ex['reference_answer'][:200]}")


## 6. ROUGE Scoring — Generated vs Reference

ROUGE-1/2/L were computed by the pipeline against `long_answer` from corpus metadata.

| Metric | Measures |
|--------|----------|
| ROUGE-1 | Unigram overlap |
| ROUGE-2 | Bigram overlap |
| ROUGE-L | Longest common subsequence |

In [ ]:
print(f"ROUGE scores over {ans_gen.get('cases')} answers:")
print(f"  ROUGE-1: {ans_gen.get('mean_rouge1') or float('nan'):.4f}")
print(f"  ROUGE-2: {ans_gen.get('mean_rouge2') or float('nan'):.4f}")
print(f"  ROUGE-L: {ans_gen.get('mean_rougeL') or float('nan'):.4f}")
print()

# Build answer_rows lookup for section 8
_corpus_meta = {doc["doc_id"]: doc.get("metadata", {}) for doc in corpus}
_trace_by_qid = {t["query_id"]: t for t in traces}
answer_rows = []
for d in details:
    pubid = str(d["query_id"]).replace("q_", "")
    t = _trace_by_qid.get(d["query_id"], {})
    retrieved_ids = [r["doc_key"] for r in t.get("retrieved", [])]
    answer_rows.append({
        "query_id":          d["query_id"],
        "question":          d["question"],
        "generated_answer":  d["generated_answer"],
        "long_answer":       d["reference_answer"],
        "final_decision":    _corpus_meta.get(pubid, {}).get("final_decision", ""),
        "retrieved_doc_ids": retrieved_ids,
        "correct_retrieved": pubid in retrieved_ids,
        "rouge1":            d["rouge1"],
        "rouge2":            d["rouge2"],
        "rougeL":            d["rougeL"],
    })

print(f"{'query_id':<20} {'R-1':>6} {'R-2':>6} {'R-L':>6} {'correct?':>9}")
print("-" * 55)
for r in answer_rows[:10]:
    r1s = f"{r['rouge1']:.4f}" if r['rouge1'] is not None else "  n/a"
    r2s = f"{r['rouge2']:.4f}" if r['rouge2'] is not None else "  n/a"
    rLs = f"{r['rougeL']:.4f}" if r['rougeL'] is not None else "  n/a"
    print(f"{r['query_id']:<20} {r1s:>6} {r2s:>6} {rLs:>6} {'checkmark' if r['correct_retrieved'] else 'x':>9}")


## 7. LLM Judge — Answer Quality

The judge ran inside the pipeline with `judge_mode="answer_quality"` — it compares the generated answer to the reference `long_answer` semantically.

Results are in `results.get_metric("llm_judge")`.

In [ ]:
llm_judge = results.get_metric("llm_judge", {})

judged = llm_judge.get("details", [])
if judged:
    pass_count = sum(1 for r in judged if r.get("judge", {}).get("verdict") == "PASS")
    print(f"Judge results over {llm_judge.get('cases')} answers:")
    print(f"  Mean score: {llm_judge.get('mean_score', 0):.4f}")
    print(f"  PASS rate:  {pass_count}/{len(judged)} = {pass_count/len(judged):.1%}")
    print()
    print(f"{'query_id':<20} {'score':>6} {'verdict':>8}")
    print("-" * 40)
    for row in judged[:10]:
        j = row.get("judge", {})
        print(f"{str(row.get('query_id', '')):<20} {j.get('score', 0):>6.3f} {j.get('verdict', ''):>8}")
    # Attach judge scores to answer_rows for section 8
    _judge_by_qid = {r["query_id"]: r.get("judge", {}) for r in judged}
    for row in answer_rows:
        j = _judge_by_qid.get(row["query_id"], {})
        row["judge_score"]   = j.get("score")
        row["judge_verdict"] = j.get("verdict")
else:
    print("No judge results (judge disabled or no traces).")
    for row in answer_rows:
        row["judge_score"] = row["judge_verdict"] = None


## 8. Does Retrieval Quality Predict Answer Quality?

If the model retrieves the correct document, does it produce a better answer?  
We compare ROUGE-L and judge score between the **hit** group (correct doc retrieved) and the **miss** group.

In [ ]:
hit_rows  = [r for r in answer_rows if r["correct_retrieved"] and r["rouge1"] is not None]
miss_rows = [r for r in answer_rows if not r["correct_retrieved"] and r["rouge1"] is not None]


def _mean(vals):
    vals = [v for v in vals if v is not None]
    return sum(vals) / len(vals) if vals else float("nan")


print("Retrieval quality vs answer quality")
print("=" * 50)
print(f"{'Group':<8} {'N':>4}  {'ROUGE-1':>8} {'ROUGE-L':>8} {'Judge':>8}")
print("-" * 50)
for label, rows in [("HIT", hit_rows), ("MISS", miss_rows)]:
    n = len(rows)
    r1 = _mean([r["rouge1"] for r in rows])
    rL = _mean([r["rougeL"] for r in rows])
    js = _mean([r.get("judge_score") for r in rows])
    print(f"{label:<8} {n:>4}  {r1:>8.4f} {rL:>8.4f} {js:>8.4f}")

print()
print(f"Total:    {len(answer_rows):>4} queries")
print(f"Hit@5:    {len(hit_rows):>4} ({len(hit_rows)/len(answer_rows):.1%})")
print(f"Miss@5:   {len(miss_rows):>4} ({len(miss_rows)/len(answer_rows):.1%})")

if hit_rows and miss_rows:
    delta_rL = _mean([r["rougeL"] for r in hit_rows]) - _mean([r["rougeL"] for r in miss_rows])
    print(f"\nROUGE-L delta (hit − miss): {delta_rL:+.4f}")
    if delta_rL > 0.02:
        print("→ Better retrieval correlates with higher answer quality.")
    elif delta_rL < -0.02:
        print("→ Retrieval quality not clearly predictive of answer quality here.")
    else:
        print("→ No significant difference — LLM may compensate for poor retrieval.")